# CSK3 int8 skip-controller test

Load the PYNQ-Z2 overlay and verify FPGA cosine statistics, reference caching, warmup and consecutive-skip control.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

overlay = Overlay('cosine_overlay.bit')
ip = overlay.cosine_skip_0
print(ip.register_map)

In [ ]:
CTRL = 0x00
AP_RETURN = 0x10
X_LOW = 0x18
LENGTH = 0x24
STEP_INDEX = 0x2C
THRESHOLD_Q15 = 0x34
WARMUP_STEPS = 0x3C
MAX_CONSECUTIVE_SKIPS = 0x44
RESET_STATE = 0x4C
DOT_OUT_LOW = 0x54
NORM_X_LOW = 0x6C
NORM_Y_LOW = 0x84
THRESHOLD_PASSED_OUT = 0x9C
SKIP_STREAK_OUT = 0xAC

def write_u64(offset, value):
    value = int(value)
    ip.write(offset, value & 0xffffffff)
    ip.write(offset + 4, (value >> 32) & 0xffffffff)

def read_u64(offset):
    return ip.read(offset) | (ip.read(offset + 4) << 32)

def read_i64(offset):
    value = read_u64(offset)
    return value - (1 << 64) if value & (1 << 63) else value

def run_controller(buf, step=0, threshold=0.999, warmup=0, max_skips=2, reset=False):
    buf.flush()
    write_u64(X_LOW, buf.physical_address)
    ip.write(LENGTH, 0 if reset else len(buf))
    ip.write(STEP_INDEX, step)
    ip.write(THRESHOLD_Q15, int(threshold * 32768))
    ip.write(WARMUP_STEPS, warmup)
    ip.write(MAX_CONSECUTIVE_SKIPS, max_skips)
    ip.write(RESET_STATE, int(reset))
    started = time.perf_counter()
    ip.write(CTRL, 1)
    while (ip.read(CTRL) & 2) == 0:
        pass
    elapsed_ms = (time.perf_counter() - started) * 1000
    dot = read_i64(DOT_OUT_LOW)
    norm_x = read_u64(NORM_X_LOW)
    norm_y = read_u64(NORM_Y_LOW)
    similarity = dot / np.sqrt(norm_x * norm_y) if norm_x and norm_y else np.nan
    return {
        'should_skip': ip.read(AP_RETURN),
        'threshold_passed': ip.read(THRESHOLD_PASSED_OUT),
        'skip_streak': ip.read(SKIP_STREAK_OUT),
        'similarity': similarity,
        'kernel_ms': elapsed_ms,
    }

In [ ]:
N = 4 * 32 * 32
feature = allocate(shape=(N,), dtype=np.int8)
rng = np.random.default_rng(1234)
feature[:] = rng.integers(-127, 128, size=N, dtype=np.int8)

print('reset:', run_controller(feature, reset=True))
print('establish:', run_controller(feature, step=0))
print('skip 1:', run_controller(feature, step=1))
print('skip 2:', run_controller(feature, step=2))
print('forced UNet:', run_controller(feature, step=3))
feature[:] = rng.integers(-127, 128, size=N, dtype=np.int8)
print('different:', run_controller(feature, step=4))